# 001 · Iterables and Iterators

> **Syntax › Iterators & Generators** · Python 3.14

Every `for` loop in Python runs on two small protocols: the **iterable** (something you can loop over) and the **iterator** (the cursor that walks through it). They map almost one-to-one onto C#'s `IEnumerable<T>` and `IEnumerator<T>`.

**In this notebook you will learn:**

- the difference between an iterable and an iterator, and how `iter()` / `next()` work
- how `StopIteration` replaces C#'s `MoveNext()` returning `false`
- how to implement `__iter__` and `__next__` in your own class
- what a `for` loop actually does under the hood
- the single-use (exhaustion) gotcha, and built-ins that consume iterables

## C# mental model

| C# | Python |
|---|---|
| `IEnumerable<T>` | **iterable** — any object with `__iter__()` |
| `IEnumerator<T>` | **iterator** — an object with `__next__()` (and `__iter__()` returning itself) |
| `GetEnumerator()` | `iter(obj)` → calls `obj.__iter__()` |
| `MoveNext()` + `Current` | `next(it)` → calls `it.__next__()` — one call does both |
| `MoveNext()` returns `false` | `__next__()` raises `StopIteration` |
| `foreach` | `for` |
| `Dispose()` on the enumerator | no equivalent on plain iterators (generators have `close()`) |

The key mental shift: Python has **no `Current` property**. `next()` both advances and returns the value, and "no more items" is signalled with an exception rather than a `bool`.

## Iterables vs iterators

An **iterable** is a collection-like thing you can loop over many times: `list`, `tuple`, `str`, `dict`, `set`, `range`, files…
Calling `iter()` on it gives you a fresh **iterator** — a one-way cursor with its own position.

```csharp
// C#
IEnumerable<int> numbers = new List<int> { 10, 20, 30 };
using IEnumerator<int> e = numbers.GetEnumerator();
e.MoveNext(); Console.WriteLine(e.Current);   // 10
e.MoveNext(); Console.WriteLine(e.Current);   // 20
```

In [ ]:
numbers = [10, 20, 30]      # iterable
it = iter(numbers)          # iterator (C#: GetEnumerator())

print(type(numbers).__name__, "->", type(it).__name__)
print(next(it))   # 10
print(next(it))   # 20
print(next(it))   # 30

## `StopIteration` instead of `MoveNext() == false`

When an iterator runs out, `next()` raises `StopIteration`. A `for` loop catches it for you — you rarely see it unless you call `next()` by hand.

In [ ]:
try:
    next(it)          # the iterator above is exhausted
except StopIteration as e:
    print(f"{type(e).__name__}: {e!r}")

Pass a **default** as the second argument and `next()` returns it instead of raising — similar to LINQ's `FirstOrDefault()` on the remaining items.

In [ ]:
it = iter(["only"])
print(next(it, "nothing left"))   # only
print(next(it, "nothing left"))   # nothing left

# Common idiom: first matching item or None (C#: FirstOrDefault(predicate))
users = ["alice", "bob", "carol"]
first_b = next((u for u in users if u.startswith("b")), None)
print(first_b)

## Telling them apart

Iterators are also iterables: `iter(an_iterator)` returns **the same object**. Collections return a **new** iterator every time. That is the practical test.

In [ ]:
from collections.abc import Iterable, Iterator

data = [1, 2, 3]
it = iter(data)

print(isinstance(data, Iterable), isinstance(data, Iterator))   # True False
print(isinstance(it, Iterable), isinstance(it, Iterator))       # True True
print(iter(data) is iter(data))   # False — new cursor each time
print(iter(it) is it)             # True  — an iterator is its own iterable

## How `for` works under the hood

```csharp
// What the C# compiler roughly generates for foreach
using (var e = items.GetEnumerator())
{
    while (e.MoveNext())
    {
        var item = e.Current;
        Console.WriteLine(item);
    }
}
```

Python's `for item in items:` is equivalent to this:

In [ ]:
items = ["a", "b", "c"]

_it = iter(items)                 # GetEnumerator()
while True:
    try:
        item = next(_it)          # MoveNext() + Current
    except StopIteration:         # MoveNext() returned false
        break
    print(item)                   # loop body

## Implementing the protocol in a class

To make your own iterator, implement two dunder methods:

- `__iter__(self)` — return the iterator (for an iterator, `return self`)
- `__next__(self)` — return the next value, or `raise StopIteration`

```csharp
// C#
class Countdown : IEnumerator<int>
{
    private int _current;
    public Countdown(int start) => _current = start + 1;
    public int Current => _current;
    object IEnumerator.Current => Current;
    public bool MoveNext() => --_current >= 1;
    public void Reset() => throw new NotSupportedException();
    public void Dispose() { }
}
```

In [ ]:
class Countdown:
    """Iterator that counts down from start to 1."""

    def __init__(self, start: int) -> None:
        self.current = start

    def __iter__(self):
        return self

    def __next__(self) -> int:
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


for n in Countdown(3):
    print(n)

`Countdown` is an *iterator*, so it can be used only once. To get a reusable **iterable** — like a C# class implementing `IEnumerable<T>` — make `__iter__` return a **new** iterator each time:

In [ ]:
class CountdownRange:
    """Iterable: every loop gets a fresh Countdown."""

    def __init__(self, start: int) -> None:
        self.start = start

    def __iter__(self):
        return Countdown(self.start)   # C#: GetEnumerator() => new Countdown(...)


r = CountdownRange(3)
print(list(r))   # [3, 2, 1]
print(list(r))   # [3, 2, 1] — works again

💡 In practice you rarely write `__next__` by hand: making `__iter__` a **generator** (using `yield`) is far shorter — exactly like using `yield return` in `GetEnumerator()` in C# *(see Iterators & Generators › 002 · Generators)*.

## ⚠️ Gotcha: iterators are single-use

Once an iterator is exhausted, it stays exhausted — there is no `Reset()`. This bites people when they pass an iterator (for example the result of `map`, `filter`, `zip` or `enumerate`) somewhere that loops over it twice.

In [ ]:
squares = map(lambda x: x * x, [1, 2, 3])   # map returns an iterator, not a list

print(list(squares))   # [1, 4, 9]
print(list(squares))   # [] — already consumed!

Another classic: an `in` check partially consumes an iterator.

In [ ]:
it = iter([1, 2, 3, 4, 5])
print(3 in it)     # True — but it consumed 1, 2, 3 to find it
print(list(it))    # [4, 5]

The fix: if you need multiple passes, materialise once with `list(...)` (C#: `.ToList()`), or keep the original iterable around and call `iter()` again.

In [ ]:
squares = list(map(lambda x: x * x, [1, 2, 3]))   # C#: .ToList()
print(sum(squares), max(squares))                 # both passes work

## Built-ins that consume iterables

Many built-ins accept **any** iterable — they call `iter()` internally, just like LINQ extension methods work on any `IEnumerable<T>`.

| C# (LINQ) | Python |
|---|---|
| `.Sum()` | `sum(xs)` |
| `.Min()` / `.Max()` | `min(xs)` / `max(xs)` |
| `.ToList()` | `list(xs)` |
| `.OrderBy(x => x).ToList()` | `sorted(xs)` |
| `.Any(pred)` | `any(pred(x) for x in xs)` |
| `.All(pred)` | `all(pred(x) for x in xs)` |
| `.ToHashSet()` / `.ToDictionary(...)` | `set(xs)` / `dict(pairs)` |

In [ ]:
prices = (19.99, 5.49, 102.00, 12.50)   # a tuple — any iterable works
words = "the quick brown fox".split()

print(sum(prices), min(prices), max(prices))
print(sorted(words, key=len))
print(any(p > 100 for p in prices))           # Any(p => p > 100)
print(all(len(w) >= 3 for w in words))        # All(w => w.Length >= 3)
print(min(words, key=len), max(words, key=len))

`min()` / `max()` on an empty iterable raise `ValueError` (C#: `InvalidOperationException` from `Min()` on an empty sequence). Use `default=`:

In [ ]:
try:
    max([])
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

print(max([], default=0))   # C#: DefaultIfEmpty(0).Max()

⚠️ **Gotcha:** iterating a `dict` yields its **keys**, not key/value pairs like C#'s `KeyValuePair<K,V>`. Use `.items()` for pairs.

In [ ]:
stock = {"apple": 3, "pear": 0}
print(list(stock))           # ['apple', 'pear']
print(list(stock.items()))   # [('apple', 3), ('pear', 0)]

## Summary

| Concept | C# | Python |
|---|---|---|
| Loopable thing | `IEnumerable<T>` | iterable (`__iter__`) |
| Cursor | `IEnumerator<T>` | iterator (`__next__`) |
| Get cursor | `GetEnumerator()` | `iter(x)` |
| Advance + read | `MoveNext()` + `Current` | `next(it)` |
| End of sequence | `MoveNext()` returns `false` | `StopIteration` raised |
| Next or default | `FirstOrDefault()` | `next(it, default)` |
| Reset | `Reset()` (rarely supported) | not possible — call `iter()` on the source again |
| Materialise | `.ToList()` | `list(it)` |
| Aggregate | `Sum`, `Min`, `Max`, `Any`, `All` | `sum`, `min`, `max`, `any`, `all` |

## Exercises

1. Write an iterator class `Cycle(items, times)` that yields the items of `items` in order, repeated `times` times (e.g. `Cycle("ab", 2)` → `a b a b`).
2. Using only `iter()` and `next()` with a default, write a function `first_two(iterable)` that returns a tuple of the first two items (missing items become `None`).
3. Why does `z = zip([1, 2], [3, 4]); print(len(list(z)), len(list(z)))` print `2 0`?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
class Cycle:
    def __init__(self, items, times):
        self.items = list(items)
        self.total = len(self.items) * times
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= self.total:
            raise StopIteration
        value = self.items[self.index % len(self.items)]
        self.index += 1
        return value

print(list(Cycle("ab", 2)))   # ['a', 'b', 'a', 'b']

# 2
def first_two(iterable):
    it = iter(iterable)
    return next(it, None), next(it, None)

print(first_two([7]))         # (7, None)

# 3 — zip() returns an iterator. The first list(z) exhausts it,
#     so the second list(z) gets nothing.
```

</details>